In [1]:
import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize

In [2]:
# Define initial state |00>
initial_state = np.array([1, 0, 0, 0], dtype=complex)

# Target Bell state = 1/sqrt2 (|00> + |11>) = [1/sqrt2, 0, 0, 1/sqrt2)
target_state = np.array([1/np.sqrt(2), 0, 0, 1/np.sqrt(2)], dtype=complex) 
print("Initial State:\n", initial_state)
print("Target State:\n", target_state)

Initial State:
 [1.+0.j 0.+0.j 0.+0.j 0.+0.j]
Target State:
 [0.70710678+0.j 0.        +0.j 0.        +0.j 0.70710678+0.j]


### Why 4x4?

A single qubit is on 2 dimensions. Since we’re applying these unitaries on a 2-qubit system, the matrix dimension is 2^2=4

In [4]:
# Define a random 4x4 Hermitian matrix
def create_random_hermitian_matrix():
    A = np.random.randn(4, 4) + 1j * np.random.randn(4, 4)
    return (A + A.conj().T) / 2  # Make it Hermitian

# Generate initial Hermitian matrices M1 and M2
M1_init = create_random_hermitian_matrix()
M2_init = create_random_hermitian_matrix()
print("Initial M1 and M2:")
M1_init, M2_init

Initial M1 and M2:


(array([[-0.12345377+0.j        ,  0.84478916-1.39814003j,
          0.26830422+0.60722632j,  0.72791655+1.02466176j],
        [ 0.84478916+1.39814003j, -0.2667326 +0.j        ,
          0.12153144-0.93938305j, -1.21909388+0.84276344j],
        [ 0.26830422-0.60722632j,  0.12153144+0.93938305j,
         -0.77454518+0.j        , -0.91321847+0.52090534j],
        [ 0.72791655-1.02466176j, -1.21909388-0.84276344j,
         -0.91321847-0.52090534j, -1.23402917+0.j        ]]),
 array([[-0.07723282+0.j        , -0.52613065-0.35591496j,
         -0.38055587-0.51781175j, -0.11578753-0.80100622j],
        [-0.52613065+0.35591496j,  0.79076697+0.j        ,
         -0.52870879-0.97578408j, -0.5986882 +0.93277945j],
        [-0.38055587+0.51781175j, -0.52870879+0.97578408j,
          0.44727298+0.j        ,  0.28768817-0.45839914j],
        [-0.11578753+0.80100622j, -0.5986882 -0.93277945j,
          0.28768817+0.45839914j, -0.88158847+0.j        ]]))

In [5]:
# Define unitaries using matrix exponentiation
def apply_unitaries(M1, M2, state):
    U1 = expm(1j * M1) # U1 = e^(iM1)
    U2 = expm(1j * M2) # U2 = e^(iM2)
    
    # Apply U1 and U2 to the state
    state_after_U1 = np.dot(U1, state)
    final_state = np.dot(U2, state_after_U1)
    return final_state

Frobenius norm: Loss=||state_generated − target_state||

In [7]:
# Define the loss function
def loss_function(params):
    # Reshape params to build M1 and M2
    M1_real, M1_imag, M2_real, M2_imag = np.split(params, 4)
    
    M1 = M1_real.reshape(4, 4) + 1j * M1_imag.reshape(4, 4)
    M2 = M2_real.reshape(4, 4) + 1j * M2_imag.reshape(4, 4)
    
    # Apply unitaries and get final state
    state_generated = apply_unitaries(M1, M2, initial_state)
    
    # Calculate loss (Frobenius norm)
    loss = np.linalg.norm(state_generated - target_state)**2
    return loss

### Why L-BFGS-B for optimization?
It's suitable for high-dimensional problems (like the initial_params-64 parameter space (16 real + 16 imag of M1 and M2)).

In [9]:
# Flatten and prepare initial guesses for M1 and M2
M1_real_init, M1_imag_init = M1_init.real.flatten(), M1_init.imag.flatten()
M2_real_init, M2_imag_init = M2_init.real.flatten(), M2_init.imag.flatten()

# Combine all parameters for optimization
initial_params = np.concatenate([M1_real_init, M1_imag_init, M2_real_init, M2_imag_init])

# Minimize the loss function using L-BFGS-B optimizer
result = minimize(loss_function, initial_params, method='L-BFGS-B', options={'maxiter': 500})

# Get optimized parameters
optimized_params = result.x
M1_opt_real, M1_opt_imag, M2_opt_real, M2_opt_imag = np.split(optimized_params, 4)

# Reconstruct optimized M1 and M2
M1_opt = M1_opt_real.reshape(4, 4) + 1j * M1_opt_imag.reshape(4, 4)
M2_opt = M2_opt_real.reshape(4, 4) + 1j * M2_opt_imag.reshape(4, 4)

print("Optimization successful:", result.success)
print("Final Loss:", result.fun)
print("Optimized M1 and M2:")
M1_opt, M2_opt

Optimization successful: True
Final Loss: 5.649537634802197e-12
Optimized M1 and M2:


(array([[ 0.04707664+0.33568887j,  0.93548834-1.57527973j,
          0.19057913+0.69406548j,  0.82355395+1.17850265j],
        [ 0.81670696+1.56766327j, -0.42333167+0.17673259j,
          0.43002989-1.01021303j, -1.19911728+0.61586263j],
        [ 0.55587864-0.35331085j, -0.21294327+0.7551716j ,
         -0.45573211+0.22677548j, -0.55419958+0.32844687j],
        [ 0.71384628-0.32788896j, -1.2708659 -1.32664703j,
         -0.8439417 -0.03874702j, -0.745387  +0.24044138j]]),
 array([[ 0.09743988+0.04903648j, -0.10340807-0.33788327j,
         -0.37626454-1.0149276j , -0.16391723-1.11280597j],
        [-0.41070849+0.66727657j,  1.12222897+0.6179875j ,
          0.42144741-1.4402998j , -0.14052306+0.29689581j],
        [-0.55110097+0.63892424j, -0.85813262+1.11592287j,
          0.67104779+0.54396314j,  0.62285813-0.12231703j],
        [-0.03510767+0.96431206j, -0.21975048-0.75772468j,
          0.49228021+0.0951979j , -0.79011147-0.23134875j]]))

### Fidelity
is the square of the magnitude of the inner product (vdot - complex conjugate dot product), representing how closely the two quantum states match.

In [11]:
# Apply the optimized unitaries to the initial state
final_state_opt = apply_unitaries(M1_opt, M2_opt, initial_state)

# Calculate fidelity with target Bell state
fidelity = np.abs(np.vdot(final_state_opt, target_state))**2
print("Optimized Final State:\n", final_state_opt)
print("Fidelity between optimized state and target Bell state:", fidelity)

Optimized Final State:
 [ 7.07108191e-01+3.86233082e-07j -3.06049247e-07+1.75592330e-07j
 -1.12742578e-06-7.73214372e-07j  7.07107759e-01-7.51522805e-07j]
Fidelity between optimized state and target Bell state: 1.0000033757473217


In [46]:
!pip install "jax[cpu]==0.4.30" "jaxlib==0.4.30" --force-reinstall

  Using cached jaxlib-0.4.30-cp39-cp39-macosx_10_14_x86_64.whl.metadata (1.0 kB)
  Using cached ml_dtypes-0.5.1-cp39-cp39-macosx_10_9_universal2.whl.metadata (21 kB)
  Using cached numpy-2.0.2-cp39-cp39-macosx_14_0_x86_64.whl.metadata (60 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached scipy-1.13.1-cp39-cp39-macosx_10_9_x86_64.whl.metadata (60 kB)
  Using cached importlib_metadata-8.6.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached zipp-3.21.0-py3-none-any.whl.metadata (3.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 10.7 MB/s eta 0:00:00
Using cached jaxlib-0.4.30-cp39-cp39-macosx_10_14_x86_64.whl (83.5 MB)
Using cached importlib_metadata-8.6.1-py3-none-any.whl (26 kB)
Using cached ml_dtypes-0.5.1-cp39-cp39-macosx_10_9_universal2.whl (667 kB)
Using cached numpy-2.0.2-cp39-cp39-macosx_14_0_x86_64.whl (6.9 MB)
Using cached scipy-1.13.1-cp39-cp39-macosx_10_9_x86_64.whl (39.4 MB)
Using cached opt_einsum-3.4.0-py3-none-any.whl (71 kB

In [50]:
# Verify installation
import jax
import jax.numpy as jnp
print(f"JAX version: {jax.__version__}")
print(f"JAXlib version: {jax.lib.__version__}")

AttributeError: module 'jax' has no attribute 'version'

In [25]:
import jax
import jax.numpy as jnp
from jax import grad, jit
from jax import random, lax
import numpy as np

ModuleNotFoundError: No module named 'jax.numpy'

In [33]:
key = random.PRNGKey(42)

# --- 1. Generate Random Initial States ---
def generate_random_initial_states(n, key):
    keys = random.split(key, n)
    def generate_single(k):
        state = random.normal(k, (4,)) + 1j * random.normal(k, (4,))
        return state / jnp.linalg.norm(state)
    return jnp.stack([generate_single(k) for k in keys])

initial_states = generate_random_initial_states(10, key)  # batch of 10 initial states

NameError: name 'random' is not defined

In [ ]:
# --- 2. Ground Truth Target (You’ll replace with real data later) ---
# Apply some fixed "true" unitary circuit to get output for supervision
def make_hermitian(M):
    return 0.5 * (M + M.conj().T)

def expm(iM):
    return jax.scipy.linalg.expm(1j * iM)

def true_circuit(state):
    M1_true = make_hermitian(random.normal(key, (4,4)) + 1j * random.normal(key, (4,4)))
    M2_true = make_hermitian(random.normal(key, (4,4)) + 1j * random.normal(key, (4,4)))
    U1 = expm(M1_true)
    U2 = expm(M2_true)
    return U2 @ (U1 @ state)

target_states = jnp.stack([true_circuit(state) for state in initial_states])

In [ ]:
# --- 3. Learnable Parameters ---
key1, key2 = random.split(key)
M1_params = random.normal(key1, (4, 4)) + 1j * random.normal(key1, (4, 4))
M2_params = random.normal(key2, (4, 4)) + 1j * random.normal(key2, (4, 4))

In [ ]:
# --- 4. Apply Learned Circuit ---
def apply_unitaries(M1, M2, state):
    M1 = make_hermitian(M1)
    M2 = make_hermitian(M2)
    U1 = expm(M1)
    U2 = expm(M2)
    return U2 @ (U1 @ state)

In [ ]:
# --- 5. Loss Function ---
def loss_fn(M1, M2, inputs, targets):
    def single_loss(xy):
        x, y = xy
        out = apply_unitaries(M1, M2, x)
        return jnp.linalg.norm(out - y) ** 2
    return jnp.mean(jax.vmap(single_loss)((inputs, targets)))

In [ ]:
# --- 6. Training Step ---
@jit
def training_step(params, inputs, targets, lr=0.01):
    M1, M2 = params
    grads = jax.grad(loss_fn, argnums=(0,1))(M1, M2, inputs, targets)
    M1_new = M1 - lr * grads[0]
    M2_new = M2 - lr * grads[1]
    return (M1_new, M2_new), loss_fn(M1_new, M2_new, inputs, targets)

In [ ]:
# --- 7. Training Loop ---
params = (M1_params, M2_params)
for epoch in range(100):
    params, loss = training_step(params, initial_states, target_states, lr=0.05)
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.6f}")